In [ ]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

import warnings

warnings.filterwarnings("ignore")
import kairo


In [ ]:
log = kairo.read_log("../data/control-flow.xes")
log = kairo.map_columns(log, {
    "case ID": "case:concept:name", "activity": "concept:name",
    "timestamp": "time:timestamp", "resource": "org:resource", "event duration": "event:duration_min",
})

In [ ]:
stats = kairo.log_statistics(log)
print(kairo.abstract_log_statistics(stats))

In [ ]:
kairo.plot_activity_frequency(stats).show()

In [ ]:
config = kairo.IntraConfig(
    features=("activity_freq", "bigram", "vocab", "progress", "current", "history"),
    clustering="som", grid=(5, 5), metric="euclidean",
    pca_components=12, window_minutes=1440 * 7,   # 3-day distribution windows
    divergence="kl",
    reference="recent",
    lookback=8
)

result = kairo.run_intra_case(log, config)
result.to_dict()

In [ ]:
kairo.plot_state_grid(result.states).show()
kairo.plot_pca_variance(result.pca).show()

In [ ]:
dist_fig = kairo.plot_state_distribution(result.distribution, result.states)
kairo.add_window_boundaries(dist_fig, result.distribution["window_start"])
dist_fig.show()
signal_fig = kairo.plot_drift_signal(result.signal, title="KL divergence vs previous window")
signal_fig.show()   # the drift point at 60% should stand out


In [ ]:
case = result.trajectories["case:concept:name"].iloc[1]
sub = result.trajectories[result.trajectories["case:concept:name"] == case]
kairo.plot_trajectory(sub["time:timestamp"], sub["state_id"].to_numpy(), result.states,
                      title=f"Case {case}").show()


In [ ]:
profiles = kairo.analysis.state_profiles(result.features, result.states)
print(kairo.abstract_states(result.states, profiles=profiles, max_len=1200))

In [ ]:
print(kairo.abstract_result(result))

In [ ]:
# Inspect the prompt without any network call
prompt = kairo.build_prompt("Where does the process drift, and what changes?",
                                result=result, log=log)
print(prompt)


In [ ]:
QUESTION = "Does this analysis show concept drift? Name the windows, the states involved, and your confidence."

In [ ]:
# Local models
print(kairo.llm.available_models("local"))

In [ ]:
answer = kairo.ask(QUESTION, result=result, log=log, executor=kairo.local_query)
print(answer)


### Hosted providers


In [ ]:
answer = kairo.ask(QUESTION, result=result, log=log,
                       executor=kairo.anthropic_query, model="claude-sonnet-5")
print(answer)


In [ ]:
# Explain a plot
explanation = kairo.explain_plot(signal_fig, "What does this drift signal show? Where would you place the drift?",
                                     result=result, executor=kairo.anthropic_query, model="claude-sonnet-5")
print(explanation)


In [ ]:
cfg = kairo.nlp_to_config(
    "look for drift at weekly granularity with DBSCAN and a fairly tight eps, cosine distance",
    perspective="intra_case", executor=kairo.anthropic_query,
)
print(cfg)
kairo.run_intra_case(log, cfg)  # …and run it
